# 滤镜、裁剪与蒙版

学习目标：能区分像素处理、可见范围、混合和文字环绕，并为视觉效果保留可读回退。

前置知识：背景与渐变、图片资源路径、透明度、正常流、浮动和层叠上下文。

适用范围：Filter Effects、CSS Masking、Compositing and Blending、CSS Shapes 的相关模块功能；backdrop-filter 与不同蒙版语法须按目标浏览器支持范围检查。不依赖 JavaScript。

环境准备：[环境配置与运行](README.md)。

配套脚本：位于 scripts/24-filters-clipping-masking/。

1. [index.html](scripts/24-filters-clipping-masking/index.html)：滤镜、裁剪、蒙版与混合。
2. [shapes.html](scripts/24-filters-clipping-masking/shapes.html)：圆形与矩形文字环绕对照。
3. [styles.css](scripts/24-filters-clipping-masking/styles.css)：各效果的独立规则。
4. [art.svg](scripts/24-filters-clipping-masking/art.svg)：自制透明底几何图案。
5. [mask.svg](scripts/24-filters-clipping-masking/mask.svg)：自制黑底白圆蒙版。

Step 1：在已激活 Python 环境的终端中，从项目根目录进入本技术目录。

```bash
cd content/Web与应用开发/css
```

Step 2：启动本章预览服务。

```bash
python -m http.server 8101 --bind 127.0.0.1
```

Step 3：打开[本章示例首页](http://127.0.0.1:8101/scripts/24-filters-clipping-masking/index.html)。

保存修改后刷新页面。

Step 4：在服务终端按 Ctrl+C 停止服务。

## 1 filter 处理元素的绘制结果

filter 对元素及其内容的绘制结果施加滤镜，可用于图像、边框和文字。本例只处理图片，保留旁边文字清楚可读。

grayscale(1) 表示完全灰度；0 表示没有灰度效果。drop-shadow() 依次给出水平偏移、垂直偏移、模糊标准差与颜色，沿输入图像的透明度轮廓生成投影；它不支持 box-shadow 的 inset 和扩展半径参数。

多个滤镜函数以空格分开，按书写顺序依次作用。blur() 接受非负长度，brightness(1) 保持原亮度、0 变黑，大于 1 变亮；它们都是 filter 的值函数，不是独立属性。

滤镜不改变正常布局尺寸，但绘制可能延伸出盒子。非 none 的 filter 还会创建层叠上下文；不要把“只改像素”理解为不影响任何定位或层叠关系。

```html
<div class="samples">
  <figure><img class="art original" src="art.svg" alt="蓝色圆形和橙色三角形" width="240" height="160"><figcaption>原图</figcaption></figure>
  <figure><img class="art filtered" src="art.svg" alt="同一几何图案" width="240" height="160"><figcaption>灰度与投影</figcaption></figure>
</div>
```

```css
.filtered { filter: grayscale(1) drop-shadow(6px 6px 3px #555); }
/* 对照原图：灰度作用于原图，后面的投影沿当前不透明轮廓产生，盒子尺寸不变。 */
```

配套文件：[index.html](scripts/24-filters-clipping-masking/index.html)、[styles.css](scripts/24-filters-clipping-masking/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/24-filters-clipping-masking/index.html)

## 2 本章使用的属性

| 完整属性名 | 中文名称／含义 | 用途或对象 |
| --- | --- | --- |
| filter | 元素滤镜 | 处理元素绘制结果 |
| backdrop-filter | 背景滤镜 | 处理元素背后的像素 |
| clip-path | 裁剪路径 | 限制可见几何范围 |
| mask | 蒙版简写 | 设置蒙版图像及相关分项 |
| mask-image | 蒙版图像 | 渐变、图像或 SVG 蒙版引用 |
| mask-mode | 蒙版解释方式 | 使用透明度或亮度 |
| mask-size | 蒙版尺寸 | 缩放蒙版图像 |
| mask-position | 蒙版位置 | 定位蒙版图像 |
| mask-repeat | 蒙版重复方式 | 是否平铺 |
| mix-blend-mode | 元素混合模式 | 元素与背后内容混合 |
| background-blend-mode | 背景混合模式 | 同一元素的背景层混合 |
| isolation | 混合隔离 | 建立隔离的混合组 |
| shape-outside | 外部环绕形状 | 浮动元素影响的文字行框 |
| shape-margin | 环绕形状外留白 | 文字与形状间距 |

circle()、linear-gradient()、grayscale() 和 url() 是值函数；alpha、luminance、multiply 和 isolate 是关键字值；@supports 是条件 @ 规则。

## 3 backdrop-filter 处理元素背后

backdrop-filter 处理元素背后的像素，不会像 filter 那样把前景文字整体模糊。要看见后面的效果，元素需要透明或半透明背景；不透明白底会把模糊结果盖住。

先给 .glass 不透明白底，再在支持 blur() 背景滤镜时改用 85% 不透明的白色背景。CSS 的 rgb() 斜杠后是 alpha 透明度，85% 越接近 100% 越不透明。

滤镜采样受到背景根（backdrop root）边界限制。例如祖先 opacity 小于 1，可能让子元素无法采样这个祖先背后的内容。效果没出现时要同时检查透明度、背后是否有可分辨内容和祖先边界，不能只检查声明是否存在。

```html
<div class="backdrop-stage">
  <p class="glass">这段文字保持清楚；观察文字后面的条纹。</p>
</div>
```

```css
.backdrop-stage {
  padding: 2rem;
  background: repeating-linear-gradient(45deg, #2e597a 0 12px, #d1e4f2 12px 24px);
}
.glass { padding: 1rem; color: #17212b; background: #fff; }
@supports (backdrop-filter: blur(6px)) {
  .glass { background: rgb(255 255 255 / 85%); backdrop-filter: blur(6px); }
}
/* 不支持时保留不透明白底；支持时透出模糊条纹，不模糊这段文字。 */
```

配套文件：[index.html](scripts/24-filters-clipping-masking/index.html)、[styles.css](scripts/24-filters-clipping-masking/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/24-filters-clipping-masking/index.html)

## 4 clip-path 限制几何可见范围

clip-path 将路径外的部分裁掉，路径内仍按原样绘制。circle(40% at 50% 50%) 定义圆，at 后给出圆心位置；默认参考 border-box，圆心百分比对应盒子宽和高，半径百分比依据参考盒的归一化对角线，不是简单取宽度的 40%。

其他常见形状包括 inset() 的内缩矩形、ellipse() 的椭圆、polygon() 的点列多边形。它们改变裁剪范围，不改变元素在正常流中占据的盒子。

裁掉的区域通常不参与指针命中，但裁剪并不等于从键盘顺序或可访问性树中删除内容。因此本例只裁剪图案，把文字和链接留在完整区域外；不要靠裁剪隐藏一个仍可聚焦的控件。

```html
<div class="clip-frame">
  <img class="art clipped" src="art.svg" alt="裁剪后的几何图案" width="240" height="160">
</div>
<p>文字与操作放在裁剪区域外：<a href="shapes.html">查看文字环绕</a>。</p>
```

```css
.clipped { clip-path: circle(40% at 50% 50%); }
/* 圆心位于参考盒中心；查看外框，裁剪不会让原有240×160盒子变小。 */
```

配套文件：[index.html](scripts/24-filters-clipping-masking/index.html)、[styles.css](scripts/24-filters-clipping-masking/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/24-filters-clipping-masking/index.html)

## 5 mask 的透明度、亮度与资源

蒙版（mask）按每个位置的蒙版值决定目标露出多少，可以产生渐变透明边缘。它与裁剪的几何边界不同，也不会改变布局盒子。

渐变蒙版通常按 alpha 通道解释：不透明黑色与不透明白色都能完整露出目标；transparent 会隐藏目标。luminance 模式还考虑亮度并与 alpha 相乘，黑色隐藏、白色露出，中间亮度产生部分透明。

mask.svg 是一张普通 SVG 图像，黑底与白圆都不透明，因此可以对照两种模式。同一 SVG 文件作为图像，与 url() 指向 SVG 内部 &lt;mask&gt; 元素不是相同来源类型；默认 match-source 对普通图像通常采用 alpha，对 SVG &lt;mask&gt; 通常采用 luminance，除非该蒙版本身另有设置。

mask 简写会重置未写出的蒙版分项，包含 mask-border 相关分项。斜杠区分位置与尺寸，no-repeat 禁止重复；后续需要单改模式时，放在简写之后。

styles.css 中 url("mask.svg") 相对于样式表 URL。蒙版图像失败或不被接受时，可能按全透明黑色处理，使被蒙版的装饰消失；不会自动退回未蒙版原图。通过本章 HTTP 服务访问，跨源资源还须满足 CORS；不要以 file 地址测试后概括网络页面行为。

```html
<div class="samples">
  <figure><img class="art fade" src="art.svg" alt="逐渐透明的几何图案" width="240" height="160"><figcaption>渐变透明度</figcaption></figure>
  <figure><img class="art mask-alpha" src="art.svg" alt="几何图案" width="240" height="160"><figcaption>同一蒙版：alpha</figcaption></figure>
  <figure><img class="art mask-luminance" src="art.svg" alt="几何图案" width="240" height="160"><figcaption>同一蒙版：luminance</figcaption></figure>
</div>
```

```css
.fade { mask: linear-gradient(to right, #000 35%, transparent) center / 100% 100% no-repeat; }
.mask-alpha, .mask-luminance {
  mask-image: url("mask.svg");
  mask-size: 100% 100%;
  mask-position: center;
  mask-repeat: no-repeat;
}
.mask-alpha { mask-mode: alpha; }
.mask-luminance { mask-mode: luminance; }
/* mask.svg 是不透明黑底白圆：alpha保留全部，luminance主要保留白圆范围。 */
```

配套文件：[index.html](scripts/24-filters-clipping-masking/index.html)、[styles.css](scripts/24-filters-clipping-masking/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/24-filters-clipping-masking/index.html)

## 6 混合模式与隔离边界

mix-blend-mode 把元素颜色与它背后的内容混合，background-blend-mode 只混合同一元素自身的背景图层与背景色。multiply 是正片叠底模式；本例用两块不同色样观察叠加，不让说明文字参与混合。

isolation: isolate 在 .blend-stage 建立隔离组，让内部混合不继续采样组外背景。它不是裁剪，也不把元素从布局中拿走。mix-blend-mode 非 normal 时会创建层叠上下文，放入现有浮层页面后还需要重新检查遮挡关系。

混合后的颜色取决于实际背后内容；同一个前景色不能保证在所有背景上都有同样对比度。必要文本用普通前景色和稳定背景，不用混合效果承载唯一状态信息。

```html
<div class="blend-stage" aria-hidden="true"><div class="blend-mark"></div></div>
<div class="background-blend" aria-hidden="true"></div>
<p>两块色样仅用于观察混合；这段说明不参与混合。</p>
```

```css
.blend-stage { isolation: isolate; background: #9ddbf5; padding: 1rem; }
.blend-mark { width: 10rem; height: 5rem; background: #f6ad55; mix-blend-mode: multiply; }
.background-blend {
  height: 6rem;
  background-color: #9ddbf5;
  background-image: linear-gradient(90deg, #f6ad55, #fff);
  background-blend-mode: multiply;
}
/* 第一例混合元素与其背后像素；第二例只混合同一元素自己的背景层。 */
```

配套文件：[index.html](scripts/24-filters-clipping-masking/index.html)、[styles.css](scripts/24-filters-clipping-masking/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/24-filters-clipping-masking/index.html)

## 7 shape-outside 改变文字环绕

shape-outside 用于浮动元素，改变旁边行内内容的环绕边界；它不会自己让元素浮动，也不会裁剪元素图像。普通 Grid 项目或没有 float 的盒子，不能仅加这一属性就获得相同环绕。

本例明确设置 160px 的宽高、float: left，并用 circle(50%) border-box 作为环绕形状。shape-margin 为形状与文字增加留白；另写 clip-path 才把图像的视觉范围也裁成圆形。

.wrap 使用 flow-root 包住浮动，第二组保持相同图像与文字但不指定形状。形状缺失时回到矩形浮动环绕，仍可阅读。图像也可以作为 shape-outside 的来源，形状由 alpha 与阈值决定；无效或跨源授权不足时不能形成预期形状。

```html
<article class="wrap shaped">
  <img class="float-art" src="art.svg" alt="用于环绕的几何图案" width="160" height="160">
  <p><span class="flow-text">这段连续文字用于观察行框如何绕开浮动图像。上方靠近圆顶的文字可以更接近图像，中间靠近圆心的行需要留出更多空间。圆形只定义环绕边界时，图像本身并不会自动被裁成圆形，所以这里另用裁剪使视觉边界一致。继续阅读可以看到图像底部以后的文字恢复整行宽度。这段连续文字用于观察行框如何绕开浮动图像。上方靠近圆顶的文字可以更接近图像，中间靠近圆心的行需要留出更多空间。圆形只定义环绕边界时，图像本身并不会自动被裁成圆形，所以这里另用裁剪使视觉边界一致。继续阅读可以看到图像底部以后的文字恢复整行宽度。</span></p>
</article>
```

```css
.wrap { display: flow-root; max-width: 36rem; line-height: 1.8; }
.wrap p { margin: 0; }
.float-art { float: left; width: 160px; height: 160px; margin: 0 1rem 0 0; object-fit: cover; }
.shaped .float-art {
  shape-outside: circle(50%) border-box;
  shape-margin: 0.75rem;
  clip-path: circle(50%);
}
/* 对照下方矩形浮动：观察每行文字起点；禁用shape-outside后恢复矩形环绕。 */
```

配套文件：[shapes.html](scripts/24-filters-clipping-masking/shapes.html)、[styles.css](scripts/24-filters-clipping-masking/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/24-filters-clipping-masking/shapes.html)

## 8 效果失效与性能观察

效果应用在图片或纯装饰色样上，标题、说明和导航保留独立可读区域。滤镜不支持时保留原图，背景滤镜不支持时保留实色；蒙版加载失败可能隐藏装饰，因此其外部说明不可依赖那张图才成立。

用开发者工具依次禁用 filter、clip-path、mask 和混合声明，检查页面信息与链接仍能使用。再在 Network 中检查 art.svg、mask.svg 的地址与响应；请求成功还需结合实际图形，不能把 200 状态当作蒙版已正确工作。

大面积模糊、复杂裁剪和随滚动变化的背景效果可能增加绘制工作，但实际成本受尺寸、浏览器和设备影响。保持同一视口与操作，在性能面板分别录制启用与关闭 .glass 的 backdrop-filter 后滚动的过程，比较绘制和帧耗时；不要凭属性名字声称必然合成加速或给出未经测量的帧率。

```html
<div class="backdrop-stage">
  <p class="glass">这段文字保持清楚；观察文字后面的条纹。</p>
</div>
```

```css
.glass { padding: 1rem; color: #17212b; background: #fff; }
```

配套文件：[index.html](scripts/24-filters-clipping-masking/index.html)、[styles.css](scripts/24-filters-clipping-masking/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/24-filters-clipping-masking/index.html)

## 本章小结

- filter 处理元素，backdrop-filter 处理背景；后者需要可见的背景像素与正确边界。
- clip-path 决定几何可见范围，mask 决定各位置透明度，两者不代替布局尺寸。
- 混合取决于背后内容，isolation 可以限制混合范围。
- shape-outside 作用于浮动的环绕，不负责浮动或图像裁剪。
- 保留可读回退，资源与性能都以真实观察为准。

## 练习

（1）在灰度前后分别添加 brightness(0.5)，并单独改变投影颜色。标准：能指出函数的处理顺序与投影出现的阶段，不能只背属性名。

（2）将 mask.svg 的黑底改为透明，比较 alpha 与 luminance。标准：说明改变的是透明度还是亮度；再在开发者工具中禁用蒙版，确认文字与链接完整。

（3）保持 shape-outside 不变，只禁用 float，再恢复 float 并禁用 clip-path。标准：分别观察环绕条件与图像可见范围，解释两种变化为何不同。

### 提示

性能练习如需比较，使用同一视口、设备和滚动过程；只报告实际录制所得。几何和颜色效果需要截图或肉眼检查，计算样式不是完整证据。

## 参考与引用来源

- W3C：[Filter Effects Level 1 的 filter 与 Filter Functions](https://www.w3.org/TR/filter-effects-1/#FilterProperty) 的作用范围、顺序与层叠上下文；[CSS Masking Level 1 §5–7](https://www.w3.org/TR/css-masking-1/#clipping-paths) 的裁剪、命中测试及蒙版；[CSS Shapes Level 1](https://www.w3.org/TR/css-shapes-1/#shape-outside-property) 的浮动、形状和资源；[Compositing and Blending Level 1](https://www.w3.org/TR/compositing-1/#isolation) 的混合与隔离。
- MDN：[filter 的 Functions](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/filter#functions)、[backdrop-filter 的 Description](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/backdrop-filter#description) 的函数与背景根；[clip-path](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/clip-path) 的参考盒；[mask-image](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/mask-image#description)、[mask-mode](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/mask-mode#values)、[mask](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/mask#description) 的 alpha、luminance、URL失败与简写重置；[mix-blend-mode](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/mix-blend-mode)、[background-blend-mode](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/background-blend-mode) 的混合对象；[shape-outside](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/shape-outside#values) 的值、浮动与 CORS。
- web.dev：[How to create high-performance CSS animations 的 Paint](https://web.dev/articles/animations-guide#paint) 与 [Layout](https://web.dev/articles/animations-guide#layout) 小节：绘制和布局成本的开发者工具观察方法。
- Python 3.12：[http.server 命令行](https://docs.python.org/3.12/library/http.server.html#command-line-interface) 的本地服务。